# Mouse ovary scRNA-seq workflow

Groups: **Y** (4-month vehicle), **OC** (10-month vehicle), **OT** (10-month MRJP1).  
Contrasts: OC vs Y = aging; OT vs OC = treatment; OT vs Y = residual deviation.

> In no-card mode, use this notebook only for metadata and small tables. Do not load the full AnnData object into memory.

In [ ]:
from pathlib import Path
import pandas as pd
import yaml
ROOT = Path('/root/autodl-tmp/ovary_scRNAseq')
with (ROOT/'config/analysis_config.yaml').open() as f:
    cfg = yaml.safe_load(f)
metadata = pd.read_csv(ROOT/'config/sample_metadata.tsv', sep='\t')
metadata

## 1. Input validation and QC review

In [ ]:
validation = ROOT/'results/00_preflight/input_validation.tsv'
qc_summary = ROOT/'results/02_qc_summary.tsv'
if validation.exists(): display(pd.read_csv(validation, sep='\t'))
if qc_summary.exists(): display(pd.read_csv(qc_summary, sep='\t'))

## 2. Atlas and annotation review

Review broad marker scores and cluster markers. Fill `config/cluster_labels.tsv`; never accept automated labels without marker support.

In [ ]:
score_path = ROOT/'results/04_cluster_marker_scores.tsv'
marker_path = ROOT/'results/04_cluster_top_markers.tsv'
if score_path.exists(): display(pd.read_csv(score_path, sep='\t').head())
if marker_path.exists(): display(pd.read_csv(marker_path, sep='\t').head(30))

## 3. Biological comparisons

Formal DE is library-level pseudobulk within each curated cell type. The rescue table combines aging, treatment, and residual contrasts.

In [ ]:
pb_root = ROOT/'results/05_pseudobulk'
rescue_tables = sorted(pb_root.glob('*/gene_rescue_summary.tsv')) if pb_root.exists() else []
[(p.parent.name, len(pd.read_csv(p, sep='\t'))) for p in rescue_tables]

## 4. Pathway-level interpretation

Prioritize ovarian support, steroidogenesis, senescence/SASP, mitochondrial function, oxidative stress, ECM/fibrosis, immune inflammation, and angiogenesis.

In [ ]:
pathway_result = ROOT/'results/06_pathways/pathway_rescue_summary.tsv'
if pathway_result.exists():
    pathway_df = pd.read_csv(pathway_result, sep='\t')
    display(pathway_df.sort_values(['cell_type', 'pathway_rescued'], ascending=[True, False]).head(50))